# The Safety-Net — a quantitative teardown 🔬
### Real total-return tape · X-sweep · HAC inference · exposure-matched placebo · Kaminski-Lo

![Signal: Weak](https://img.shields.io/badge/Signal-Weak-dab617?style=flat-square)
![Tradability: Mirage](https://img.shields.io/badge/Tradability-Mirage-c0392b?style=flat-square)
![Cuts drawdown?: Mixed](https://img.shields.io/badge/Cuts_drawdown%3F-Mixed-dab617?style=flat-square)

The deep companion to the [notebook for the curious](01_for_the_curious.ipynb) — *same seven beats, every claim now carrying its standard error.* We separate the two things a trailing stop does — **reduce drawdown** (real, but only in a width band) and **forecast return** (not certifiable) — and show the result is exactly the **Kaminski & Lo (2014)** dichotomy: stops help trending tapes and hurt mean-reverting ones, and a long-biased index sits in the unhelpful zone.

> ⚠️ **Not investment advice.** SPY daily, total-return adjusted (`quantlab.data`, Yahoo); cash earns 0% (conservative); 5 bps/switch; 21-day re-entry cooldown; one-day execution lag. Sources in [`docs/references.md`](../docs/references.md), reproducible run in [`docs/results.md`](../docs/results.md).
>
> 💡 **The `💡 In plain words` notes** translate each result back into intuition.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))          # study root (safety_net/)
sys.path.insert(0, os.path.abspath("../../.."))    # repo root (quantlab/)
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (10, 5.5)
import numpy as np, pandas as pd
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
from safety_net import data, strategy

AS_OF = "2026-06-12"
COOLDOWN, COST = 21, 5.0
STOPS = (5.0, 10.0, 15.0, 20.0)
frame = data.load_real("SPY", mode="total_return").loc[:AS_OF]
close = frame["close"]
bh = strategy.buy_and_hold(close)
runs = {x: strategy.backtest(close, strategy.trailing_stop_position(close, x, COOLDOWN), cost_bps=COST) for x in STOPS}
stop10 = runs[10.0]
print(f"SPY total return: {len(close):,} rows  {close.index[0].date()} -> {close.index[-1].date()}  fingerprint={data.fingerprint(frame)}")
print(f"B&H        Sharpe {bh['sharpe']:.3f}  maxDD {bh['max_dd']*100:.1f}%  CAGR {bh['cagr']*100:.2f}%")
print(f"stop 10%   Sharpe {stop10['sharpe']:.3f}  maxDD {stop10['max_dd']*100:.1f}%  CAGR {stop10['cagr']*100:.2f}%  TiM {stop10['time_in_market']*100:.0f}%")


SPY total return: 8,400 rows  1993-01-29 -> 2026-06-12  fingerprint=d74f157c34ef
B&H        Sharpe 0.646  maxDD -55.2%  CAGR 10.82%
stop 10%   Sharpe 0.752  maxDD -43.9%  CAGR 10.80%  TiM 92%


## Verdict, up front

| Axis | Stamp | Why |
|---|---|---|
| **Signal** | `WEAK` | No threshold out-earns buy-and-hold: best is 15% at **+0.03 pts/yr**, 10% matches at **-0.02**, 5% is **-4.77**. HAC *t* on the daily return difference runs **-2.66 (5%) to -0.39 (10%)** — never a significant positive edge. The 10/15% stops beat a matched random-exit coin 200/200, so the timing isn't pure noise — but it doesn't pay. |
| **Tradability** | `MIRAGE` | As a *return improver* it fails: best case equals B&H, tight stops destroy return (4.8 pts/yr at 5%). Any Sharpe gain is lower beta, obtainable more cheaply by holding less stock. |
| **Cuts drawdown?** | `MIXED` | maxDD cut is **11 pts at 10%** but only **0 pts at 20%**; the 5% stop cuts drawdown yet whipsaws return. Protection is genuine only in a middle width band. |

> 💡 **In plain words:** the stop-loss is a *drawdown reducer that works at one dial setting* wearing the costume of a *return booster that doesn't exist*.

## 1 · The claim, steelmanned

- **H₁ (risk):** a trailing stop lowers drawdown and volatility vs buy-and-hold.
- **H₂ (skill):** the stop's *exit dates* beat an exposure-matched random schedule.
- **H₃ (the sold claim):** the stop *improves returns* / risk-adjusted return.

Kaminski & Lo (2014): the stopping premium has the **sign of the return autocorrelation** — positive (helps) for trending assets, negative (hurts) for mean-reverting ones. A long-biased index with short-horizon mean reversion should land near or below zero on H₃.

## 2 · So what? — what rides on each answer

If H₃ held, the simplest risk rule on Earth would be a return engine too. It doesn't — and seeing *why* (the V-shaped rebounds the stop sells into) is the Kaminski-Lo lesson made concrete.

## 3 · How we'd know — the protocol

Sweep X over 5/10/15/20% · HAC inference on each return difference · exposure-matched random-exit placebo at each X · the trending-vs-mean-reverting synthetic controls that pin the theory.

## 4 · The teardown

The full X-sweep, all net of 5 bps/switch:

In [2]:
tbl = pd.DataFrame({
  'CAGR %':  [bh['cagr']*100] + [runs[x]['cagr']*100 for x in STOPS],
  'Vol %':   [bh['vol']*100]  + [runs[x]['vol']*100 for x in STOPS],
  'Sharpe':  [bh['sharpe']]   + [runs[x]['sharpe'] for x in STOPS],
  'MaxDD %': [bh['max_dd']*100]+ [runs[x]['max_dd']*100 for x in STOPS],
  'TiM %':   [bh['time_in_market']*100] + [runs[x]['time_in_market']*100 for x in STOPS],
}, index=['Buy & hold'] + [f'Stop {int(x)}%' for x in STOPS])
tbl.round(3)

,CAGR %,Vol %,Sharpe,MaxDD %,TiM %
Buy & hold,10.817,18.577,0.646,-55.189,100.000
Stop 5%,6.051,12.598,0.530,-48.042,75.738
Stop 10%,10.801,15.184,0.752,-43.881,92.238
Stop 15%,10.847,16.153,0.719,-46.690,95.738
Stop 20%,10.046,16.942,0.650,-54.726,97.738


**HAC inference + matched random-exit placebo** at each threshold — is any edge real, and does the timing beat random?

In [3]:
rows = []
for x in STOPS:
    pos = strategy.trailing_stop_position(close, x, COOLDOWN)
    st = runs[x]
    diff = (st['net']-bh['net']).to_numpy()
    t = strategy.hac_tstat(diff)
    rand = np.array([strategy.backtest(close, strategy.matched_random_position(pos, seed=s), cost_bps=COST)['sharpe'] for s in range(200)])
    rows.append({'X %': int(x), 'Sharpe-B&H': st['sharpe']-bh['sharpe'], 'HAC t (ret diff)': t,
                 'beats coin /200': int((st['sharpe']>rand).sum()), 'CAGR gap pts': (st['cagr']-bh['cagr'])*100,
                 'DD cut pts': (st['max_dd']-bh['max_dd'])*100})
pd.DataFrame(rows).set_index('X %').round(3)

,Sharpe-B&H,HAC t (ret diff),beats coin /200,CAGR gap pts,DD cut pts
X %,,,,,
5,-0.116,-2.659,93,-4.766,7.147
10,0.106,-0.389,200,-0.016,11.308
15,0.073,-0.289,200,0.030,8.499
20,0.004,-1.058,156,-0.771,0.464


> 💡 **In plain words:** at 10–15% the stop's exits land in genuinely better spots than random (beats the coin 200/200) — but the *return* difference's HAC *t* is near zero or negative, so it never out-earns the market. The 5% stop has a *significantly negative* edge (t = -2.66): too tight, it whipsaws.

**The Kaminski-Lo dichotomy, on synthetic tapes** — the machinery proof that the engine banks the edge where it should and not where it shouldn't:

In [4]:
trend, _ = data.synthetic_trending(n_days=6000, seed=99)
mr, _ = data.synthetic_meanrev(n_days=6000, seed=99)
for label, fr in [('TRENDING (stops help)', trend), ('MEAN-REVERTING (stops hurt)', mr)]:
    c = fr['close']
    st = strategy.backtest(c, strategy.trailing_stop_position(c, 10.0, COOLDOWN), cost_bps=COST)
    b = strategy.buy_and_hold(c)
    win = 'stop WINS' if st['sharpe']>b['sharpe'] else 'stop LOSES'
    print(f"{label:30s}  stop Sharpe {st['sharpe']:.3f}  vs  B&H {b['sharpe']:.3f}   -> {win}")

TRENDING (stops help)           stop Sharpe 0.596  vs  B&H 0.422   -> stop WINS
MEAN-REVERTING (stops hurt)     stop Sharpe 0.207  vs  B&H 0.428   -> stop LOSES


> 💡 **In plain words:** plant a persistent downtrend and the stop escapes it (wins); plant mean reversion and the stop sells every dip into the rebound (loses). SPY behaves like the second case — which is why the real stop doesn't pay. *(Synthetic Sharpes are a positive control for the harness, never market evidence for the stamp.)*

## 5 · The verdict

Signal `WEAK` (no width out-earns B&H; HAC *t* on the return diff -2.66 to -0.39; 10/15% beat the matched coin 200/200), Tradability `MIRAGE` (best case equals B&H, tight stops lose 4.8 pts/yr), Cuts drawdown? `MIXED` (11 pts at 10% vs 0 pts at 20%). The drawdown cut is lower exposure; the return claim is the mean-reversion the index keeps handing back.

## 6 · Could you trade it?

Capacity is a non-issue (SPY). The binding facts are economic: (1) the strategy delivers **equal or less return for less risk** — a lower-beta book a 90/10 stock/cash blend reproduces without dozens of taxable switches; and (2) **width risk is real** — a plausibly-chosen 5% stop costs 4.8 pts/yr. Net of tax the gap to buy-and-hold only widens.

## 7 · Going further

- Replace 0% cash with the **T-bill** (`^IRX`) and recompute excess-of-cash for both arms.
- A **volatility-targeted** trailing stop (X in ATR / realised-vol units) vs the fixed %.
- Test the engine on **trending** assets (managed-futures, single-name momentum) where Kaminski-Lo predict the stop *should* earn its keep — does the desk's harness confirm it live?